# HSTransform v0.2.0 — Validation Notebook

Tests all signals from the original `power_quality_disturbance` and `fault_analysis` notebooks using the corrected algorithm.

**What changed in v0.2.0 (relevant here):**
- Window is now computed as the proper DFT of the hyperbolic Gaussian (complex-valued, phase included)
- FFT slice off-by-one fixed (`h[k:k+n]` instead of `h[k+1:k+n+1]`)
- `np.sum(g)` scalar bug fixed — window is now a vector
- DC row alternating-pattern bug fixed

**Note on curvature:** Default `curvature=312.5` activates hyperbolic asymmetry for `|t| ≳ √312.5 ≈ 17.7`.  
For `t ∈ [0, 10]` (power quality signals below), the asymmetry is marginal.  
Use `curvature=0.25` for unit-normalised time axes to get full asymmetric behaviour.

In [ ]:
# Install from local repo (v0.2.0 with all algorithm fixes)
import subprocess, sys
repo = __import__('pathlib').Path('..').resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo), '--quiet'])
print('HSTransform installed from:', repo)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from hstransform import HSTransform

print('HSTransform import OK')

## Matplotlib style
Preserved from the original notebooks.

In [ ]:
plt.rcParams['figure.facecolor'] = '#F0F8FF'
plt.rcParams['figure.titlesize'] = 'medium'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.edgecolor'] = 'Blue'
plt.rcParams['figure.frameon'] = True
plt.rcParams['figure.autolayout'] = True

plt.rcParams['axes.facecolor'] = '#F5F5DC'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'normal'
plt.rcParams['axes.titlecolor'] = 'Olive'
plt.rcParams['axes.edgecolor'] = 'pink'
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.titlelocation'] = 'center'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['axes.labelcolor'] = 'Olive'
plt.rcParams['axes.axisbelow'] = False
plt.rcParams['axes.xmargin'] = 0.2
plt.rcParams['axes.ymargin'] = 0.2

plt.rcParams['xtick.bottom'] = True
plt.rcParams['xtick.color'] = '#A52A2A'
plt.rcParams['ytick.left'] = True
plt.rcParams['ytick.color'] = '#A52A2A'

plt.rcParams['grid.color'] = 'green'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['grid.alpha'] = 0.3

plt.rcParams['legend.loc'] = 'right'
plt.rcParams['legend.facecolor'] = 'NavajoWhite'
plt.rcParams['legend.edgecolor'] = 'pink'
plt.rcParams['legend.shadow'] = True
plt.rcParams['legend.fontsize'] = 10

plt.rcParams['font.size'] = 12

print('Style applied')

---
## Part 1 — Power Quality Disturbances

Four voltage disturbance types on a 50 Hz, 220 V (peak × √2) signal:
- **Sag**: 50% of nominal for 1.5 s
- **Swell**: 150% of nominal for 2 s
- **Spike**: 500% of nominal for 0.2 s
- **Notch**: 0 V for 0.3 s

Source: `power_quality_disturbance-Copy1.ipynb`

In [ ]:
t = np.linspace(0, 10, 100)   # same time axis as original notebooks
V_m = 220 * np.sqrt(2)
f_V = 50

V_clean = V_m * np.sin(2 * np.pi * f_V * t)

V_sag   = np.where((t >= 3) & (t <= 4.5), 0.5 * V_clean, V_clean)
V_swell = np.where((t >= 3) & (t <= 5.0), 1.5 * V_clean, V_clean)
V_spike = np.where((t >= 3) & (t <= 3.2), 5.0 * V_clean, V_clean)
V_notch = np.where((t >= 3) & (t <= 3.3), 0.0 * V_clean, V_clean)

# Disturbance window positions in scaled time (0.1*t)
highlight = [
    (0.30, 0.45),   # sag
    (0.30, 0.50),   # swell
    (0.30, 0.32),   # spike
    (0.30, 0.33),   # notch
]

signals = {
    'Sag\n(50% for 1.5 s)':   V_sag,
    'Swell\n(150% for 2 s)':  V_swell,
    'Spike\n(500% for 0.2 s)': V_spike,
    'Notch\n(0 V for 0.3 s)': V_notch,
}

st = HSTransform()
wavelet = 'morl'
scales = np.arange(1, 101)

print('Signals ready. Transformer:', st)

### 1.1 Spectrogram comparison: HST vs Morlet Wavelet

In [ ]:
fig, axs = plt.subplots(3, len(signals), figsize=(21, 11), sharex=True)
fig.suptitle('Power Quality Disturbances — HST vs Morlet Wavelet', fontsize=14, color='Olive')

for i, (name, signal) in enumerate(signals.items()):
    S = st.fit_transform(t, signal)
    coeffs, freqs = pywt.cwt(signal, scales, wavelet)
    t_plot = 0.1 * t

    # Row 0: raw signal
    axs[0, i].plot(t_plot, signal, color='steelblue', lw=1.5)
    axs[0, i].set_title(f'Voltage — {name}', fontsize=12)
    axs[0, i].axvspan(*highlight[i], color='orange', alpha=0.4, label='disturbance')
    if i == 0:
        axs[0, i].legend(fontsize=9)

    # Row 1: HS-transform power spectrum
    im1 = axs[1, i].imshow(
        np.abs(S)**2, aspect='auto', cmap='jet',
        extent=[t_plot.min(), t_plot.max(), 0, S.shape[1]]
    )
    axs[1, i].set_title('HST Power Spectrum', fontsize=12)
    axs[1, i].axvspan(*highlight[i], color='orange', alpha=0.2)
    if i == 0:
        axs[1, i].set_ylabel('Frequency bin', fontsize=10)

    # Row 2: Morlet CWT power spectrum
    im2 = axs[2, i].imshow(
        np.abs(coeffs)**2, aspect='auto', cmap='jet',
        extent=[t_plot.min(), t_plot.max(), freqs.min(), freqs.max()]
    )
    axs[2, i].set_title('Morlet Wavelet Power', fontsize=12)
    axs[2, i].axvspan(*highlight[i], color='orange', alpha=0.2)
    axs[2, i].set_xlabel('Time (s)', fontsize=10)
    if i == 0:
        axs[2, i].set_ylabel('Scale (pseudo-freq)', fontsize=10)

plt.savefig('power_quality_disturbance.png', dpi=150)
plt.show()

### 1.2 Real / Imaginary trajectory of the S-transform

In [ ]:
freq_list = [5, 10, 30, 50]
fig, axs = plt.subplots(len(freq_list) + 2, len(signals), figsize=(21, 24), sharex=False)
fig.suptitle('Power Quality — HST Real/Imaginary Trajectories', fontsize=14, color='Olive')

for i, (name, signal) in enumerate(signals.items()):
    S = st.fit_transform(t, signal)

    # Row 0: raw signal
    axs[0, i].plot(0.1 * t, signal, color='steelblue', lw=1.5)
    axs[0, i].set_title(f'Voltage — {name}', fontsize=11)
    axs[0, i].axvspan(*highlight[i], color='orange', alpha=0.4)

    # Row 1: aggregate real/imag power trajectory
    axs[1, i].plot(
        np.sum(S.imag**2, axis=0),
        np.sum(S.real**2, axis=0),
        lw=1.2, color='teal'
    )
    axs[1, i].set_title('Imag² vs Real²\n(all frequencies)', fontsize=11)
    axs[1, i].set_xlabel('Σ Imag²', fontsize=9)
    axs[1, i].set_ylabel('Σ Real²', fontsize=9)

    # Rows 2+: per-frequency trajectories
    for j, freq in enumerate(freq_list):
        axs[j + 2, i].plot(
            S[freq, :].imag, S[freq, :].real,
            lw=1.2, color='darkorange'
        )
        axs[j + 2, i].set_title(f'Trajectory at {freq} Hz', fontsize=11)
        axs[j + 2, i].set_xlabel('Imag', fontsize=9)
        if i == 0:
            axs[j + 2, i].set_ylabel('Real', fontsize=9)

plt.savefig('power_quality_disturbance_trajectory.png', dpi=150)
plt.show()

---
## Part 2 — Power System Fault Analysis

Synthetic 3-phase current signals (Ia, Ib, Ic) for four fault scenarios:

| Case | Description |
|---|---|
| Healthy | Balanced 3-phase, no fault |
| SLGF A | Single line-to-ground, Phase A |
| DLGF A-B | Double line-to-ground, Phases A and B |
| 3-phase | Symmetric three-phase fault |

Source: `fault_analysis.ipynb` (synthetic equivalents — original required `test_fault_dataset.csv`)

In [ ]:
n_samples = 450
max_t = 0.04   # 2 cycles at 50 Hz (same as original: t = linspace(0, 0.04, 450))
t_fault = np.linspace(0, max_t, n_samples)
f0 = 50
I_peak = 500   # nominal peak current (A)
fault_start = int(0.4 * n_samples)   # fault at 40% into the window

def make_3phase(t, peak=I_peak):
    """Balanced 3-phase sinusoids."""
    Ia = peak * np.sin(2 * np.pi * f0 * t)
    Ib = peak * np.sin(2 * np.pi * f0 * t - 2 * np.pi / 3)
    Ic = peak * np.sin(2 * np.pi * f0 * t + 2 * np.pi / 3)
    return Ia, Ib, Ic

Ia0, Ib0, Ic0 = make_3phase(t_fault)

# SLGF Phase A: Ia collapses to near 0, Ib/Ic show imbalance transient
Ia_slgf = Ia0.copy()
Ia_slgf[fault_start:] *= 0.1
Ib_slgf = Ib0 + 0.3 * I_peak * np.where(t_fault >= t_fault[fault_start], np.sin(2*np.pi*150*t_fault), 0)
Ic_slgf = Ic0 + 0.3 * I_peak * np.where(t_fault >= t_fault[fault_start], np.sin(2*np.pi*150*t_fault + np.pi), 0)

# DLGF A-B: both Ia and Ib collapse
Ia_dlgf = Ia0.copy(); Ia_dlgf[fault_start:] *= 0.05
Ib_dlgf = Ib0.copy(); Ib_dlgf[fault_start:] *= 0.05
Ic_dlgf = Ic0 * 1.8   # Ic increases to carry fault current

# 3-phase fault: all three collapse with high-frequency transient
def fault_transient(t, start_idx, f_hf=250, decay=500):
    x = np.zeros_like(t)
    idx = np.arange(start_idx, len(t))
    dt = t[idx] - t[start_idx]
    x[idx] = 0.5 * I_peak * np.exp(-decay * dt) * np.sin(2*np.pi*f_hf*t[idx])
    return x

Ia_3ph = Ia0 * np.where(t_fault >= t_fault[fault_start], 0.1, 1.0) + fault_transient(t_fault, fault_start)
Ib_3ph = Ib0 * np.where(t_fault >= t_fault[fault_start], 0.1, 1.0) + fault_transient(t_fault, fault_start, f_hf=250, decay=400)
Ic_3ph = Ic0 * np.where(t_fault >= t_fault[fault_start], 0.1, 1.0) + fault_transient(t_fault, fault_start, f_hf=250, decay=600)

fault_data = {
    'Healthy\n(No Fault)':             {'Ia': Ia0,    'Ib': Ib0,    'Ic': Ic0},
    'SLGF\n(Phase A)':                  {'Ia': Ia_slgf,'Ib': Ib_slgf,'Ic': Ic_slgf},
    'DLGF\n(Phase A-B)':                {'Ia': Ia_dlgf,'Ib': Ib_dlgf,'Ic': Ic_dlgf},
    '3-Phase Fault\n(A-B-C)':          {'Ia': Ia_3ph, 'Ib': Ib_3ph, 'Ic': Ic_3ph},
}

print('Fault scenarios ready.')

### 2.1 Spectrogram comparison: HST vs Morlet Wavelet (with additive noise)

In [ ]:
noise_level = 0.25
signal_phase = 'Ia'
freq_list_fault = [10, 50, 100, 200]

fig, axs = plt.subplots(3, len(fault_data), figsize=(24, 11), sharex=True)
fig.suptitle(f'Fault Analysis — Phase {signal_phase[-1]} — HST vs Morlet Wavelet', fontsize=14, color='Olive')

for i, (title, phases) in enumerate(fault_data.items()):
    signal = phases[signal_phase].copy()

    # Add harmonic + white noise (same as fault_analysis.ipynb)
    if i > 0:
        harmonic = noise_level * I_peak * (
            np.sin(2*np.pi*3*f0*t_fault) +
            np.sin(2*np.pi*5*f0*t_fault) +
            np.sin(2*np.pi*7*f0*t_fault)
        )
        white = 0.4 * I_peak * np.random.default_rng(42 + i).normal(0, noise_level, len(signal))
        signal = signal + harmonic + white

    S = st.fit_transform(t_fault, signal)
    coeffs, freqs = pywt.cwt(signal, scales, wavelet)

    axs[0, i].plot(t_fault * 1000, signal, lw=1.2, color='steelblue', label=signal_phase)
    axs[0, i].plot(t_fault * 1000, phases['Ib'], lw=1.0, alpha=0.6, label='Ib')
    axs[0, i].plot(t_fault * 1000, phases['Ic'], lw=1.0, alpha=0.6, label='Ic')
    axs[0, i].set_title(title, fontsize=11)
    axs[0, i].axvline(t_fault[fault_start] * 1000, color='red', lw=1.5, ls='--', alpha=0.7, label='fault')
    if i == 0:
        axs[0, i].legend(fontsize=8)
        axs[0, i].set_ylabel('Current (A)', fontsize=10)

    axs[1, i].imshow(
        np.abs(S)**2, aspect='auto', cmap='jet',
        extent=[t_fault.min()*1000, t_fault.max()*1000, 0, S.shape[1]]
    )
    axs[1, i].axvline(t_fault[fault_start]*1000, color='white', lw=1.5, ls='--', alpha=0.8)
    axs[1, i].set_title(f'HST Power Spectrum ({signal_phase})', fontsize=11)
    if i == 0:
        axs[1, i].set_ylabel('Frequency bin', fontsize=10)

    axs[2, i].imshow(
        np.abs(coeffs)**2, aspect='auto', cmap='jet',
        extent=[t_fault.min()*1000, t_fault.max()*1000, freqs.min(), freqs.max()]
    )
    axs[2, i].axvline(t_fault[fault_start]*1000, color='white', lw=1.5, ls='--', alpha=0.8)
    axs[2, i].set_title(f'Morlet Wavelet Power ({signal_phase})', fontsize=11)
    axs[2, i].set_xlabel('Time (ms)', fontsize=10)
    if i == 0:
        axs[2, i].set_ylabel('Scale (pseudo-freq)', fontsize=10)

plt.savefig('fault_current.png', dpi=150)
plt.show()

### 2.2 Real / Imaginary trajectory per fault type

In [ ]:
freq_list_fault = [10, 50, 100, 200]

fig, axs = plt.subplots(len(freq_list_fault) + 2, len(fault_data), figsize=(24, 24), sharex=False)
fig.suptitle('Fault Analysis — HST Real/Imaginary Trajectories', fontsize=14, color='Olive')

for i, (title, phases) in enumerate(fault_data.items()):
    signal = phases[signal_phase]
    S = st.fit_transform(t_fault, signal)

    # Raw signal (all 3 phases)
    axs[0, i].plot(t_fault * 1000, phases['Ia'], lw=1.2, label='Ia')
    axs[0, i].plot(t_fault * 1000, phases['Ib'], lw=1.0, alpha=0.7, label='Ib')
    axs[0, i].plot(t_fault * 1000, phases['Ic'], lw=1.0, alpha=0.7, label='Ic')
    axs[0, i].set_title(title, fontsize=11)
    axs[0, i].axvline(t_fault[fault_start]*1000, color='red', lw=1.5, ls='--', alpha=0.7)
    if i == 0:
        axs[0, i].legend(fontsize=8)
        axs[0, i].set_ylabel('Current (A)', fontsize=10)

    # Aggregate power trajectory
    axs[1, i].plot(
        np.sum(S.imag**2, axis=0),
        np.sum(S.real**2, axis=0),
        lw=1.2, color='teal'
    )
    axs[1, i].set_title('Imag² vs Real²\n(all frequencies)', fontsize=10)
    axs[1, i].set_xlabel('Σ Imag²', fontsize=9)
    if i == 0:
        axs[1, i].set_ylabel('Σ Real²', fontsize=9)

    # Per-frequency trajectories
    for j, freq in enumerate(freq_list_fault):
        axs[j + 2, i].plot(
            S[freq, :].imag, S[freq, :].real,
            lw=1.2, color='darkorange'
        )
        axs[j + 2, i].set_title(f'Trajectory at {freq} Hz', fontsize=10)
        axs[j + 2, i].set_xlabel('Imag', fontsize=9)
        if i == 0:
            axs[j + 2, i].set_ylabel('Real', fontsize=9)

plt.savefig('fault_trajectory.png', dpi=150)
plt.show()

---
## Part 3 — Asymmetric window demo

Shows the hyperbolic asymmetry in action: an impulse at t=0 with `forwardtaper ≠ backwardtaper`.  
Uses `curvature=0.01` so asymmetry is active for the time scale used here.

In [ ]:
n_imp = 512
t_imp = np.linspace(-1, 1, n_imp)
impulse = np.zeros(n_imp)
impulse[n_imp // 2] = 1.0

lf, lb = 0.4, 0.1   # forward taper 4× wider than backward

st_sym  = HSTransform(forwardtaper=0.25, backwardtaper=0.25, curvature=0.01)
st_asym = HSTransform(forwardtaper=lf,   backwardtaper=lb,   curvature=0.01)

S_sym  = st_sym.fit_transform(t_imp,  impulse, fsamplingrate=5)
S_asym = st_asym.fit_transform(t_imp, impulse, fsamplingrate=5)

row = 10   # frequency row to inspect

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Asymmetric vs Symmetric Window — Impulse Response', color='Olive')

axs[0].plot(t_imp, impulse, color='steelblue', lw=1.5)
axs[0].set_title('Input: unit impulse at t=0')
axs[0].set_xlabel('Time')

axs[1].plot(t_imp, np.abs(S_sym[row, :]),  lw=1.5, label=f'Symmetric (λf=λb=0.25)', color='teal')
axs[1].plot(t_imp, np.abs(S_asym[row, :]), lw=1.5, label=f'Asymmetric (λf={lf}, λb={lb})', color='darkorange', ls='--')
axs[1].set_title(f'|S(τ, {row})| — magnitude at freq row {row}')
axs[1].set_xlabel('Time')
axs[1].legend()

mid = n_imp // 2
axs[2].semilogy(t_imp[mid:],  np.abs(S_sym[row,  mid:]) + 1e-10, lw=1.5, label='Symmetric', color='teal')
axs[2].semilogy(t_imp[mid:],  np.abs(S_asym[row, mid:]) + 1e-10, lw=1.5, label='Asym forward', color='darkorange')
axs[2].semilogy(-t_imp[:mid][::-1], np.abs(S_asym[row, :mid])[::-1] + 1e-10,
                lw=1.5, ls='--', color='purple', label='Asym backward')
axs[2].set_title('Decay profile (log scale)\nasym forward vs backward')
axs[2].set_xlabel('|Time from impulse|')
axs[2].legend()

plt.show()
print(f'λf={lf} (forward/anti-causal), λb={lb} (backward/causal).')
print('Larger λ → slower decay → wider window side.')